In [1]:
import os
import numpy as np
import torch

from cosmos_predict2.utils.printer import print_batch
from cosmos_predict2.utils.vis_helpers import save_action_as_image
from imaginaire.utils.io import save_image_or_video

from imaginaire.lazy_config import LazyCall as L
from cosmos_predict2.data.action_conditioned.uha_dataset import OxeUhaDataModule, NoEncoder

2025-10-14 22:45:32.015641: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-10-14 22:45:32.052165: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-10-14 22:45:32.052191: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-10-14 22:45:32.053686: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-10-14 22:45:32.061077: I tensorflow/core/platform/cpu_feature_guar

In [2]:
transforms_conf = dict(
    move_axis=True,
    bytes_to_string=True,
    adjust_type=None,
    add_robot_information=False
)

language_encoders_conf = dict(
    model_name=None
)

frame_transform_kwargs=dict(
    image_augment_kwargs=dict(
        primary=dict(
            random_resized_crop=dict(
                scale=(0.8, 1.0),
                ratio=(0.9, 1.1)
            ),
            random_brightness=(0.1,),
            random_contrast=(0.9, 1.1),
            random_saturation=(0.9, 1.1),
            random_hue=(0.05,),
            augment_order=(
                "random_resized_crop",
                "random_brightness",
                "random_contrast",
                "random_saturation",
                "random_hue",
            ),
        ),
        secondary=dict(
            random_resized_crop=dict(
                scale=(0.8, 1.0),
                ratio=(0.9, 1.1)
            ),
            random_brightness=(0.1,),
            random_contrast=(0.9, 1.1),
            random_saturation=(0.9, 1.1),
            random_hue=(0.05,),
            augment_order=(
                "random_resized_crop",
                "random_brightness",
                "random_contrast",
                "random_saturation",
                "random_hue",
            ),
        ),
        wrist=dict(
            random_brightness=(0.1,),
            random_contrast=(0.9, 1.1),
            random_saturation=(0.9, 1.1),
            random_hue=(0.05,),
            augment_order=(
                "random_brightness",
                "random_contrast",
                "random_saturation",
                "random_hue",
            ),
        ),
    ),
    resize_size=dict(
        primary=(176, 176),
        secondary=(160, 160),  # not used
        wrist=(84, 84),  # all black
    ),
    resize_size_future_obs=dict(
        primary=(176, 176),
        secondary=(160, 160),  # should be same as resize_size
        wrist=(84, 84),
    ),
    num_parallel_calls=6,
)


DEBUG_DATASET = "bridge2"
DEBUG_DATASET_MAPPING = {
    "fractal": "fractal",
    "bridge2": "bridge",
}

n_v_cond, n_v_out = 4 * 1 + 1, 4 * 5  # 4+1+20=25
n_a_out = n_v_out
n_latent_v_cond, n_latent_v_out = 1 * 1 + 1, 1 * 5  # 1+1+5=7
horizon = n_v_cond + n_v_out # 25
pad_before = n_v_cond - 1
datasets_conf = dict(
    DATA_NAME="bridge",  # ori: "fractal"
    DATA_PATH="/home/geyuan/local_soft/huggingface/v1/",
    load_camera_views=["primary"],  # ori: ["primary", "secondary", "wrist"],
    load_proprio=True,  # ori: False
    load_language_embeddings=True,  # ori: False
    action_proprio_normalization_type="bounds",
    interleaved_dataset_cfg=dict(
        shuffle_buffer_size=5000,  # ori: 5000
        balance_weights=True,
        traj_transform_kwargs=dict(
            goal_relabeling_strategy=None,
            goal_relabeling_kwargs=dict(
                min_bound=20,
                max_bound=50,
                frame_diff=3
            ),
            window_size=n_v_cond,
            action_horizon=n_a_out,
            skip_unlabeled=True,
            load_future_frames=True, # NOTE: ori: False
        ),
        frame_transform_kwargs=frame_transform_kwargs,
        traj_transform_threads=16,
        traj_read_threads=8,
    )
)

uha_datamodule = OxeUhaDataModule(
    transforms=transforms_conf,
    language_encoders=language_encoders_conf,
    datasets=datasets_conf,
    batch_size=2,
    drop_last=True,
    # CosmosPredict2 specific
    use_ori_uha_data_collate=False,
    p_camera_drop=0.,
    p_proprio_drop=0.,
    state_t=n_latent_v_cond + n_latent_v_out,
)
# uha_datamodule.prepare_data()
# uha_datamodule.setup()

train_dataloader = uha_datamodule.train_dataloader()

[DEBUG] OxeUhaDataModule prepare_data finished.
[DEBUG] Loading language embeddings from: /home/geyuan/local_soft/huggingface/v1/lang_emb_t5xxl/bridge/t5_embeddings.npz
[DEBUG] Created static lookup with 19974 embeddings, shape: (512, 1024)


2025-10-14 22:47:59.800822: I tensorflow/core/grappler/optimizers/data/replicate_on_split.cc:32] Running replicate on split optimization
[WARNING  | tensorflow         ]: AutoGraph could not transform <function _gcd_import at 0x7fda485cf400> and will run it as-is.
Cause: Unable to locate the source code of <function _gcd_import at 0x7fda485cf400>. Note that functions defined in certain environments, like the interactive Python shell, do not expose their source code. If that is the case, you should define them in a .py source file. If you are certain the code is graph-compatible, wrap the call using @tf.autograph.experimental.do_not_convert. Original error: could not get source code
To silence this warning, decorate the function with @tf.autograph.experimental.do_not_convert


Cause: Unable to locate the source code of <function _gcd_import at 0x7fda485cf400>. Note that functions defined in certain environments, like the interactive Python shell, do not expose their source code. If that is the case, you should define them in a .py source file. If you are certain the code is graph-compatible, wrap the call using @tf.autograph.experimental.do_not_convert. Original error: could not get source code
To silence this warning, decorate the function with @tf.autograph.experimental.do_not_convert


2025-10-14 22:48:02.341072: I tensorflow/core/grappler/optimizers/data/replicate_on_split.cc:32] Running replicate on split optimization



######################################################################################
# Loading the following 1 datasets (incl. sampling weight):                         #
# bridge_dataset: ==========================================================1.000000 #
######################################################################################

[DEBUG] Loading language embeddings from: /home/geyuan/local_soft/huggingface/v1/lang_emb_t5xxl/bridge/t5_embeddings.npz
[DEBUG] Created static lookup with 19974 embeddings, shape: (512, 1024)


2025-10-14 22:50:00.475416: I tensorflow/core/grappler/optimizers/data/replicate_on_split.cc:32] Running replicate on split optimization


[DEBUG] OxeUhaDataModule setup finished (main=True). Train len=1097763, Info: {'train_dataset': {'bridge_dataset': {'action': {'mean': array([ 2.17586829e-04,  1.25083316e-04, -1.71082705e-04, -1.61711127e-04,
       -2.52483500e-04,  2.51576508e-04,  5.87948561e-01]), 'std': array([0.00963237, 0.01350065, 0.01251056, 0.02814523, 0.03028245,
       0.0758547 , 0.48771718]), 'max': array([0.41691166, 0.25864795, 0.21218234, 3.12220192, 1.86181128,
       6.28047848, 1.        ]), 'min': array([-0.40075102, -0.13874775, -0.225539  , -3.20107865, -1.86181128,
       -6.27907562,  0.        ]), 'p99': array([0.02812228, 0.04063032, 0.03994889, 0.08121916, 0.07724379,
       0.2021405 , 1.        ]), 'p01': array([-0.02853955, -0.04143204, -0.02597738, -0.08020887, -0.0921306 ,
       -0.20548619,  0.        ]), 'mask': array([ True,  True,  True,  True,  True,  True, False])}, 'num_transitions': array(2195527), 'num_trajectories': array(60064), 'proprio': {'mean': array([ 0.30904329,  0.03

In [3]:
from tqdm import tqdm

vis_idx = 0
in_sample = None

for idx, batch in enumerate(tqdm(train_dataloader)):
    if idx == 0:
        print_batch(DEBUG_DATASET, batch)
    if idx < vis_idx:
        continue
    print_batch(DEBUG_DATASET, batch)
    # print("language:", batch["task"]["language_instruction"])

    in_sample = batch
    break

  0%|                                                                                             | 0/1097763 [00:00<?, ?it/s]WARNING: All log messages before absl::InitializeLog() is called are written to STDERR
W0000 00:00:1760441825.310766 2249521 op_level_cost_estimator.cc:699] Error in PredictCost() for the op: op: "CropAndResize" attr { key: "T" value { type: DT_FLOAT } } attr { key: "extrapolation_value" value { f: 0 } } attr { key: "method" value { s: "bilinear" } } inputs { dtype: DT_FLOAT shape { dim { size: 1 } dim { size: 176 } dim { size: 176 } dim { size: -127 } } } inputs { dtype: DT_FLOAT shape { dim { size: -2 } dim { size: 4 } } } inputs { dtype: DT_INT32 shape { dim { size: -2 } } } inputs { dtype: DT_INT32 shape { dim { size: 2 } } } device { type: "CPU" vendor: "GenuineIntel" model: "111" frequency: 2100 num_cores: 192 environment { key: "cpu_instruction_set" value: "AVX SSE, SSE2, SSE3, SSSE3, SSE4.1, SSE4.2" } environment { key: "eigen" value: "3.4.90" } l1_cache

bridge2: Dict, keys=['action', 'video', 'agent_pos', 'annotation_file', '__key__', 't5_text_embeddings', 't5_text_mask', 'fps', 'image_size', 'num_frames', 'padding_mask', 'sample_n_views', 'view_indices', 'latent_view_indices_B_T']
--action, <class 'torch.Tensor'>, shape=torch.Size([2, 25, 7]), min=-1.0000, max=0.8389
--video, <class 'torch.Tensor'>, shape=torch.Size([2, 3, 25, 176, 176]), min=0.0000, max=255.0000
--agent_pos, <class 'torch.Tensor'>, shape=torch.Size([2, 25, 7]), min=-0.6617, max=0.7584
--annotation_file: <class 'str'>, len=4, value='None'
--__key__: <class 'str'>, len=4, value='None'
--t5_text_embeddings, <class 'torch.Tensor'>, shape=torch.Size([2, 512, 1024]), min=-0.5859, max=0.5781
--t5_text_mask, <class 'torch.Tensor'>, shape=torch.Size([512]), min=1.0000, max=1.0000
--fps, <class 'torch.Tensor'>, shape=torch.Size([2]), min=10.0000, max=10.0000
--image_size, <class 'torch.Tensor'>, shape=torch.Size([4]), min=176.0000, max=176.0000
--num_frames: <class 'int'>, va

  0%|                                                                                             | 0/1097763 [01:00<?, ?it/s]


In [4]:
""" Visualization Remapped Dataloader """
max_vis_len = 50
mv_sample = in_sample


horizon = mv_sample['action'][0].shape[0]
save_image_or_video(
    mv_sample['video'][2, :horizon].float() / 255.,
    f"/home/geyuan/code/cospred2nvidia/output/de_{DEBUG_DATASET}_mv_agentview.mp4",
    fps=4
)

if DEBUG_DATASET == "fractal":
    meta_p01 = [-0.22453528, -0.14820013, -0.23158971, -0.35179949, -0.41930113, -0.43643461,  0.        ]
    meta_p99 = [0.17824687, 0.1493838 , 0.21842355, 0.5892666 , 0.35272657, 0.44796681, 1.        ]
elif DEBUG_DATASET == "bridge2":
    meta_p01 = [-0.02853955, -0.04143204, -0.02597738, -0.08020887, -0.0921306 , -0.20548619,  0.        ]
    meta_p99 = [0.02812228, 0.04063032, 0.03994889, 0.08121916, 0.07724379, 0.2021405 , 1.        ]

def denorm_action(act):
    return (act + 1) / 2 * (np.array(meta_p99) - np.array(meta_p01)) + np.array(meta_p01)

save_action_as_image(
    denorm_action(mv_sample['action'])[:, :3],
    f"/home/geyuan/code/cospred2nvidia/output/de_{DEBUG_DATASET}_mv_action012.png",
)
save_action_as_image(
    denorm_action(mv_sample['action'])[:, 6:7],
    f"/home/geyuan/code/cospred2nvidia/output/de_{DEBUG_DATASET}_mv_action6.png",
)

save_action_as_image(
    mv_sample['agent_pos'][:, :3],
    f"/home/geyuan/code/cospred2nvidia/output/de_{DEBUG_DATASET}_mv_agentpos012.png",
)
# save_action_as_image(
#     mv_sample['agent_pos'][:, -1:] * dataset_multi_view.meta_gripper_states_std[-1] + dataset_multi_view.meta_gripper_states_mean[-1],
#     "/home/geyuan/code/cospred2nvidia/output/de_{DEBUG_DATASET}_mv_agentpos6.png",
# )


IndexError: index 2 is out of bounds for dimension 0 with size 2

In [5]:
""" Visualization Original Dataloader """
max_vis_len = 50

def save_view(in_sample_, batch_key_: str, view_key_: str):
    in_video_ = in_sample_[batch_key_][f'image_{view_key_}']  # (B,T,C,H,W)
    in_video_ = in_video_.permute(0, 2, 1, 3, 4)  # (B,C,T,H,W)
    suffix = "_future" if "future" in batch_key_ else ""
    save_image_or_video(
        in_video_[:, :max_vis_len].to(torch.float32) / 255.,  # (c,t,h,w)
        f"/home/geyuan/code/cospred2nvidia/output/de_{DEBUG_DATASET}_in_{view_key_}{suffix}.mp4",
        fps=10
    )

# 1. Views
save_view(in_sample, 'observation', 'primary')
save_view(in_sample, 'observation', 'secondary')
save_view(in_sample, 'observation', 'wrist')

save_view(in_sample, 'future_frames', 'primary')
save_view(in_sample, 'future_frames', 'secondary')
save_view(in_sample, 'future_frames', 'wrist')

# 2. Actions
in_action = in_sample['action'][:, -1]  # (B,T,H,D) -> (B,H,D), in [-1,1]
save_action_as_image(
    in_action[0, :max_vis_len, :3],
    f"/home/geyuan/code/cospred2nvidia/output/de_{DEBUG_DATASET}_in_normed_action.png",
)

if DEBUG_DATASET == "fractal":
    meta_p01 = [-0.22453528, -0.14820013, -0.23158971, -0.35179949, -0.41930113, -0.43643461,  0.        ]
    meta_p99 = [0.17824687, 0.1493838 , 0.21842355, 0.5892666 , 0.35272657, 0.44796681, 1.        ]
elif DEBUG_DATASET == "bridge2":
    meta_p01 = [-0.02853955, -0.04143204, -0.02597738, -0.08020887, -0.0921306 , -0.20548619,  0.        ]
    meta_p99 = [0.02812228, 0.04063032, 0.03994889, 0.08121916, 0.07724379, 0.2021405 , 1.        ]
in_action_unnormed = (in_action + 1) / 2 * (np.array(meta_p99) - np.array(meta_p01)) + np.array(meta_p01)
save_action_as_image(
    in_action_unnormed[0, :max_vis_len, :3],
    f"/home/geyuan/code/cospred2nvidia/output/de_{DEBUG_DATASET}_in_unnormed_action.png",
)

# 3. Language
in_language = in_sample['task']['language_instruction'][0]
in_lang_emb = in_sample['task']['language_embedding'][0]
print(in_language)
print(in_lang_emb[0, :10])

data = np.load(f"/home/geyuan/local_soft/huggingface/v1/lang_emb_t5xxl/{DEBUG_DATASET}/t5_embeddings.npz", allow_pickle=True)
map_text_2_emb = data['text_to_embedding_map'].item()
print(map_text_2_emb[in_language][0, :10])

# 4. Proprio
in_proprio = in_sample['observation']['proprio']  # (B,T,D), in [-1,1]
dataset_meta = uha_datamodule.dataset_info
subdataset_meta = list(dataset_meta['train_dataset'].values())[0]
meta_proprio_p01 = subdataset_meta['proprio']['p01']
meta_proprio_p99 = subdataset_meta['proprio']['p99']
in_proprio_unnormed = (in_proprio + 1) / 2 * (np.array(meta_proprio_p99) - np.array(meta_proprio_p01)) + np.array(meta_proprio_p01)
save_action_as_image(
    in_proprio_unnormed[0, :max_vis_len, :3],
    f"/home/geyuan/code/cospred2nvidia/output/de_{DEBUG_DATASET}_in_unnormed_proprio.png",
)


KeyError: 'observation'